# 03 — Model Training
**Training and comparing 4 regression algorithms**

I trained two rounds of experiments:
- **Experiment 1** — all 18 features including pollutants
- **Experiment 2** — 12 weather/time features only (no pollutant leakage)


## Setup — imports and data

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

# Load and preprocess
df = pd.read_csv("../data/delhi_ncr_aqi_dataset.csv")
df = df.drop(columns=['datetime','date','station','latitude','longitude'])

day_map    = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6}
season_map = {'monsoon':0,'summer':1,'post_monsoon':2,'winter':3}
df['day_of_week'] = df['day_of_week'].map(day_map)
df['season']      = df['season'].map(season_map)
df['city']        = LabelEncoder().fit_transform(df['city'])

y_reg = df['aqi']

print("Data ready.")

Data ready.


## Experiment 1 — All features (18 columns)
Includes pollutant readings as input features alongside weather data.


In [2]:
X_all = df.drop(columns=['aqi','aqi_category'])
X_train, X_test, y_train, y_test = train_test_split(X_all, y_reg, test_size=0.2, random_state=42)

lr  = LinearRegression()
dt  = DecisionTreeRegressor(random_state=42)
rf  = RandomForestRegressor(n_estimators=100, random_state=42)
xgb = XGBRegressor(n_estimators=100, random_state=42)

for model in [lr, dt, rf, xgb]:
    model.fit(X_train, y_train)

print(f"{'Model':<22} {'Train R²':>10} {'Test R²':>10} {'MAE':>8}")
print("-"*53)
for name, model in [('Linear Regression',lr),('Decision Tree',dt),('Random Forest',rf),('XGBoost',xgb)]:
    tr = r2_score(y_train, model.predict(X_train))
    te = r2_score(y_test,  model.predict(X_test))
    mae= mean_absolute_error(y_test, model.predict(X_test))
    print(f"{name:<22} {tr:>10.4f} {te:>10.4f} {mae:>8.2f}")

Model                    Train R²    Test R²      MAE
-----------------------------------------------------
Linear Regression          0.9416     0.9411    33.51
Decision Tree              1.0000     1.0000     0.00
Random Forest              1.0000     1.0000     0.00
XGBoost                    1.0000     1.0000     0.51


**Finding:** All models score near-perfectly (R² ≈ 1.0) because pollutant columns
are the direct inputs used to calculate AQI — not genuine prediction.
This is data leakage. I moved to Experiment 2 to address this.


## Experiment 2 — Weather features only (12 columns)
Removed all pollutant columns. The model must predict AQI from weather
and temporal context alone — a genuinely useful real-world scenario.


In [3]:
weather_features = ['year','month','day','hour',
                    'day_of_week','is_weekend','season','city',
                    'temperature','humidity','wind_speed','visibility']

X_w = df[weather_features]
X_tr, X_te, y_tr, y_te = train_test_split(X_w, y_reg, test_size=0.2, random_state=42)

lr2  = LinearRegression()
dt2  = DecisionTreeRegressor(random_state=42)
rf2  = RandomForestRegressor(n_estimators=100, random_state=42)
xgb2 = XGBRegressor(n_estimators=100, random_state=42)

for model in [lr2, dt2, rf2, xgb2]:
    model.fit(X_tr, y_tr)

print(f"{'Model':<22} {'Train R²':>10} {'Test R²':>10} {'MAE':>8}")
print("-"*53)
for name, model in [('Linear Regression',lr2),('Decision Tree',dt2),('Random Forest',rf2),('XGBoost',xgb2)]:
    tr  = r2_score(y_tr, model.predict(X_tr))
    te  = r2_score(y_te, model.predict(X_te))
    mae = mean_absolute_error(y_te, model.predict(X_te))
    print(f"{name:<22} {tr:>10.4f} {te:>10.4f} {mae:>8.2f}")

Model                    Train R²    Test R²      MAE
-----------------------------------------------------
Linear Regression          0.9243     0.9233    37.66
Decision Tree              1.0000     0.9480    26.24
Random Forest              0.9962     0.9724    20.32
XGBoost                    0.9769     0.9740    20.05


**Results:**
| Model | Train R² | Test R² | MAE |
|-------|----------|---------|-----|
| Linear Regression | 0.9243 | 0.9233 | 37.66 |
| Decision Tree | 1.0000 | 0.9480 | 26.24 |
| Random Forest | 0.9962 | 0.9724 | 20.32 |
| **XGBoost** | **0.9769** | **0.9740** | **20.05** |

**Winner: XGBoost** — best test R², lowest MAE, smallest gap between train and test scores.
Decision Tree shows clear overfitting (Train 1.0 → Test 0.948).


## Save best model

In [4]:
joblib.dump(xgb2, '../models/best_model_xgboost.pkl')
print("Model saved to models/best_model_xgboost.pkl")

Model saved to models/best_model_xgboost.pkl
